In [ ]:
# =========================
# 0) Install & Drive (Colab)
# =========================
!pip install -q --no-input transformers datasets accelerate peft bitsandbytes scikit-learn rapidfuzz matplotlib

from google.colab import drive
drive.mount('/content/drive')



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.5 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# =========================
# 1) Imports (as-is) + seed
# =========================
import os, json, re, random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from datasets import Dataset
from collections import defaultdict  # (keep as original; not used now)
from typing import Dict, Any

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)
# from sklearn.model_selection import train_test_split as sk_train_test_split  # (not needed now)
import torch.nn.functional as F
from rapidfuzz.fuzz import ratio as fuzz_ratio  # (kept for parity)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


In [ ]:
# =========================
# 2) Paths (Prepared splits only)
# =========================
# 👉 তোমার প্রস্তুত CSVs (Review, Label) — split করা রেডি ডেটা


PREP_DIR  = "/content/drive/MyDrive/NEW/Dataset/prepared"  # <-- CHANGE THIS
TRAIN_CSV = f"{PREP_DIR}/train.csv"
VAL_CSV   = f"{PREP_DIR}/val.csv"
TEST_CSV  = f"{PREP_DIR}/test.csv"

MODEL_SAVE_PATH = "/content/drive/MyDrive/NEW/culturax-base-3b-seqcls"
MODEL_NAME      = "BanglaLLM/BanglaLLama-3.2-3b-unlop-culturax-base-v0.0.3"

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

TEXT_COL = "Review"


In [ ]:
# =========================
# 3) Load prepared splits (no preprocess / no balancing)
# =========================
def normalize_text(s: str) -> str:
    s = str(s).strip()
    return re.sub(r"\s+", " ", s)

def _load_ready_csv(path):
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns and "Label" in df.columns, f"{path} needs {TEXT_COL}, Label"
    df = df[[TEXT_COL, "Label"]].dropna().copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str).map(normalize_text)
    df = df[(df[TEXT_COL]!="") & df["Label"].isin([0,1])]
    return df.reset_index(drop=True)

train_df = _load_ready_csv(TRAIN_CSV)
val_df   = _load_ready_csv(VAL_CSV)
test_df  = _load_ready_csv(TEST_CSV)

print(f"Loaded → TRAIN={len(train_df)} | VAL={len(val_df)} | TEST={len(test_df)}")
print("Class counts:", {
    "train": train_df["Label"].value_counts().to_dict(),
    "val":   val_df["Label"].value_counts().to_dict(),
    "test":  test_df["Label"].value_counts().to_dict(),
})

# HF datasets (mirror your flow)
train_raw = Dataset.from_pandas(train_df)
val_raw   = Dataset.from_pandas(val_df)
test_raw  = Dataset.from_pandas(test_df)

# Keep originals for reports
val_texts  = val_raw[TEXT_COL]
val_labels = val_raw["Label"]
test_texts = test_raw[TEXT_COL]
test_labels= test_raw["Label"]


Loaded → TRAIN=10004 | VAL=1251 | TEST=1251
Class counts: {'train': {0: 5006, 1: 4998}, 'val': {0: 626, 1: 625}, 'test': {0: 626, 1: 625}}


In [ ]:
# =========================
# 4) Tokenizer (pad fix) + map
# =========================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False, trust_remote_code=True)

tokenizer.padding_side = "right"
ADDED_PAD_TOKEN = False
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    else:
        tokenizer.add_special_tokens({"pad_token": "<pad>"})
        ADDED_PAD_TOKEN = True
PAD_TOKEN_ID = tokenizer.pad_token_id
assert PAD_TOKEN_ID is not None

MAX_LEN = 256

def tok_fn(examples):
    toks = tokenizer(
        examples[TEXT_COL],
        truncation=True, max_length=MAX_LEN, padding=False,
        return_attention_mask=True,
    )
    toks["labels"] = [int(x) for x in examples["Label"]]
    return toks

train_ds = train_raw.map(tok_fn, batched=True, remove_columns=train_raw.column_names)
val_ds   = val_raw.map(tok_fn,   batched=True, remove_columns=val_raw.column_names)
test_ds  = test_raw.map(tok_fn,  batched=True, remove_columns=test_raw.column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

Map:   0%|          | 0/10004 [00:00<?, ? examples/s]

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

Map:   0%|          | 0/1251 [00:00<?, ? examples/s]

In [ ]:
# =========================
# 5) 4-bit + LoRA (as-is)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.pad_token_id = PAD_TOKEN_ID
if ADDED_PAD_TOKEN:
    model.resize_token_embeddings(len(tokenizer))

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

model.print_trainable_parameters()


config.json:   0%|          | 0.00/885 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.25G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at BanglaLLM/BanglaLLama-3.2-3b-unlop-culturax-base-v0.0.3 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 4,593,664 || all params: 3,217,349,632 || trainable%: 0.1428


In [ ]:
# =========================
# 6) Collator, metrics (same)
# =========================
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": pr, "recall": rc, "f1": f1}


In [ ]:
# =========================
# 7) TrainingArguments & Trainer (unchanged hyperparams)
# =========================
training_args = TrainingArguments(
    output_dir=MODEL_SAVE_PATH,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    remove_unused_columns=False,
)

# NOTE: তোমার আগের স্ক্রিপ্টে eval_dataset=test_ds ছিল; তাই এখানেও test_ds রেখেছি।
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,   # keep parity with your Colab
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-3099541497.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# =========================
# 8) Resume (Drive) + Train + Save
# =========================
resume_ckpt = None
if os.path.isdir(MODEL_SAVE_PATH):
    cks = [d for d in os.listdir(MODEL_SAVE_PATH) if d.startswith("checkpoint-")]
    if cks:
        cks_sorted = sorted(cks, key=lambda x: int(x.split("-")[-1]))
        resume_ckpt = os.path.join(MODEL_SAVE_PATH, cks_sorted[-1])
        print(f"🔁 Resuming from: {resume_ckpt}")
    else:
        print("🆕 No checkpoint found; training from scratch.")
else:
    print("🆕 Output dir not found; will create.")

trainer.train(resume_from_checkpoint=resume_ckpt)

final_dir = os.path.join(MODEL_SAVE_PATH, "final")
os.makedirs(final_dir, exist_ok=True)
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"✅ Final model saved to: {final_dir}")


🔁 Resuming from: /content/drive/MyDrive/NEW/culturax-base-3b-seqcls/checkpoint-939


Step,Training Loss


✅ Final model saved to: /content/drive/MyDrive/NEW/culturax-base-3b-seqcls/final


In [ ]:
# =========================
# 9) Evaluate (HF) + Save predictions (VAL + TEST)
# =========================
# HF evaluate (on test_ds per your setup)
print("📊 Eval (HF):", trainer.evaluate())

def _save_preds(ds, texts, y_true, prefix):
    out    = trainer.predict(ds)
    logits = out.predictions
    y_true = np.array(y_true, dtype=int)
    y_pred = np.argmax(logits, axis=1)
    probs  = F.softmax(torch.tensor(logits), dim=1).cpu().numpy()

    pred_df = pd.DataFrame({
        "Review": texts,
        "True":   y_true,
        "Pred":   y_pred,
        "Prob_Fake(0)":    probs[:,0],
        "Prob_NonFake(1)": probs[:,1],
    })
    pred_df.to_csv(os.path.join(MODEL_SAVE_PATH, f"{prefix}_predictions_with_probs.csv"),
                   index=False, encoding="utf-8")

    rep = classification_report(y_true, y_pred, labels=[0,1],
                                target_names=["Fake(0)", "Non-Fake(1)"], digits=4, zero_division=0)
    with open(os.path.join(MODEL_SAVE_PATH, f"{prefix}_classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(rep)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    pd.DataFrame(cm, index=["True_Fake(0)","True_NonFake(1)"],
                 columns=["Pred_Fake(0)","Pred_NonFake(1)"]
    ).to_csv(os.path.join(MODEL_SAVE_PATH, f"{prefix}_confusion_matrix.csv"), encoding="utf-8")

    # optional AUC
    try:
        auc_nonfake = roc_auc_score(y_true, probs[:,1])
        auc_fake    = roc_auc_score(1 - y_true, probs[:,0])
        with open(os.path.join(MODEL_SAVE_PATH, f"{prefix}_auc.json"), "w", encoding="utf-8") as f:
            json.dump({"AUC_NonFake(1)": float(auc_nonfake), "AUC_Fake(0)": float(auc_fake)},
                      f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"{prefix} AUC skipped:", e)

    # threshold sweep (class 1)
    ths = np.linspace(0.30, 0.70, 9)
    rows = []
    for t in ths:
        y_hat = (probs[:,1] >= t).astype(int)
        acc = accuracy_score(y_true, y_hat)
        pr1, rc1, f1_1, _ = precision_recall_fscore_support(y_true, y_hat, average="binary", pos_label=1, zero_division=0)
        cm_t = confusion_matrix(y_true, y_hat, labels=[0,1])
        rows.append({"threshold": t, "acc": acc, "prec_1": pr1, "rec_1": rc1, "f1_1": f1_1,
                     "FN_nonfake": int(cm_t[1,0]), "FP_nonfake": int(cm_t[0,1])})
    pd.DataFrame(rows).to_csv(os.path.join(MODEL_SAVE_PATH, f"{prefix}_threshold_tuning_nonfake.csv"),
                              index=False, encoding="utf-8")

# Save both VAL & TEST predictions/reports
_save_preds(val_ds,  val_texts,  val_labels,  "val")
_save_preds(test_ds, test_texts, test_labels, "test")

print("✅ All artifacts written under:", MODEL_SAVE_PATH)


📊 Eval (HF): {'eval_loss': 0.12207657843828201, 'eval_accuracy': 0.9856115107913669, 'eval_precision': 0.9856560877144382, 'eval_recall': 0.9856115107913669, 'eval_f1': 0.9856111246347647, 'eval_runtime': 152.2663, 'eval_samples_per_second': 8.216, 'eval_steps_per_second': 1.031, 'epoch': 3.0}


✅ All artifacts written under: /content/drive/MyDrive/NEW/culturax-base-3b-seqcls


In [ ]:
# =========================
# 10) Print saved VAL/TEST reports to console (no re-compute)
# =========================
import os, json
import pandas as pd
import numpy as np

# Set your save path (use your actual MODEL_SAVE_PATH if different)
MODEL_SAVE_PATH = "/content/drive/MyDrive/NEW/culturax-base-3b-seqcls"

def _print_section(title):
    print("\n" + "="*10 + f" {title} " + "="*10)

def _print_split(split):
    # split: "val" or "test"
    cr_path  = os.path.join(MODEL_SAVE_PATH, f"{split}_classification_report.txt")
    cm_path  = os.path.join(MODEL_SAVE_PATH, f"{split}_confusion_matrix.csv")
    auc_path = os.path.join(MODEL_SAVE_PATH, f"{split}_auc.json")
    th_path  = os.path.join(MODEL_SAVE_PATH, f"{split}_threshold_tuning_nonfake.csv")
    pred_path= os.path.join(MODEL_SAVE_PATH, f"{split}_predictions_with_probs.csv")

    _print_section(f"{split.upper()} • Classification Report")
    if os.path.exists(cr_path):
        with open(cr_path, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("classification_report.txt not found:", cr_path)

    _print_section(f"{split.upper()} • Confusion Matrix")
    if os.path.exists(cm_path):
        cm_df = pd.read_csv(cm_path, index_col=0)
        print(cm_df.to_string())
    else:
        print("confusion_matrix.csv not found:", cm_path)

    _print_section(f"{split.upper()} • AUC (if available)")
    if os.path.exists(auc_path):
        with open(auc_path, "r", encoding="utf-8") as f:
            print(json.load(f))
    else:
        print("AUC file not found (optional).")

    _print_section(f"{split.upper()} • Best threshold for class=1 by F1")
    if os.path.exists(th_path):
        th_df = pd.read_csv(th_path)
        best = th_df.sort_values("f1_1", ascending=False).iloc[0]
        print(best.to_string())
    else:
        print("threshold_tuning_nonfake.csv not found (optional).")

    _print_section(f"{split.upper()} • Top-5 confident errors")
    if os.path.exists(pred_path):
        df = pd.read_csv(pred_path)
        errs = df[df["True"] != df["Pred"]].copy()
        if len(errs) == 0:
            print("No misclassifications.")
        else:
            # Confidence of the predicted class
            conf = np.where(errs["Pred"]==0, errs["Prob_Fake(0)"], errs["Prob_NonFake(1)"])
            errs["PredConfidence"] = conf
            cols = ["Review","True","Pred","Prob_Fake(0)","Prob_NonFake(1)","PredConfidence"]
            print(errs.sort_values("PredConfidence", ascending=False).head(5)[cols].to_string(index=False))
    else:
        print("predictions_with_probs.csv not found.")

# Run for both splits
_print_split("val")
_print_split("test")



========== VAL • Classification Report ==========
              precision    recall  f1-score   support

     Fake(0)     0.9719    0.9936    0.9826       626
 Non-Fake(1)     0.9935    0.9712    0.9822       625

    accuracy                         0.9824      1251
   macro avg     0.9827    0.9824    0.9824      1251
weighted avg     0.9827    0.9824    0.9824      1251


========== VAL • Confusion Matrix ==========
                 Pred_Fake(0)  Pred_NonFake(1)
True_Fake(0)              622                4
True_NonFake(1)            18              607

========== VAL • AUC (if available) ==========
{'AUC_NonFake(1)': 0.9978926517571886, 'AUC_Fake(0)': 0.997891373801917}

========== VAL • Best threshold for class=1 by F1 ==========
threshold      0.300000
acc            0.984013
prec_1         0.993475
rec_1          0.974400
f1_1           0.983845
FN_nonfake    16.000000
FP_nonfake     4.000000

========== VAL • Top-5 confident errors ==========
                                